# Test Fine-Tuned Legal-BERT Dual-Head Model

In this notebook, we load the fine-tuned dual-head model and test it on a raw ToS clause.

In [5]:
import torch
import torch.nn as nn
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

MODEL_DIR = Path("../../saved_models/lawgic_classifier_legal-bert_v3").resolve()
TOPICS_JSON = MODEL_DIR / "lawgic_topics_44.json"

NUM_LAWGIC_TOPICS = 44
NUM_HARM_CLASSES = 3

HARM_CLASS_NAMES = {0: "Harmful", 1: "Neutral", 2: "Fair"}

In [6]:
# Load Topic Mapping
with TOPICS_JSON.open("r", encoding="utf-8") as f:
    topics_data = json.load(f)
    
id2label = {t["classifier_id"]: t["topic_id"] for t in topics_data}
id2name = {t["classifier_id"]: t["name"] for t in topics_data}
print(f"Loaded {len(id2label)} topics.")

Loaded 44 topics.


In [7]:
class LawgicDualHeadModel(nn.Module):
    def __init__(self, model_name: str, num_topics: int = NUM_LAWGIC_TOPICS, num_harm_classes: int = NUM_HARM_CLASSES):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.topic_head = nn.Linear(hidden_size, num_topics)
        self.harm_head = nn.Linear(hidden_size, num_harm_classes)
        self.num_topics = num_topics
        self.num_harm_classes = num_harm_classes

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        cls_embedding = outputs.pooler_output
        topic_logits = self.topic_head(cls_embedding)
        harm_logits = self.harm_head(cls_embedding)
        return topic_logits, harm_logits

In [8]:
# Initialize tokenizer and model
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# For the encoder, we load from MODEL_DIR. 
model = LawgicDualHeadModel(str(MODEL_DIR))

# Load the full state dict which includes the heads.
state_dict_path = MODEL_DIR / "model_state_dict.pt"
if state_dict_path.exists():
    model.load_state_dict(torch.load(state_dict_path, map_location="cpu", weights_only=True))
else:
    # Fallback to loading heads individually if needed
    topic_weights = torch.load(MODEL_DIR / "topic_head_weights.pt", map_location="cpu", weights_only=True)
    harm_weights = torch.load(MODEL_DIR / "harm_head_weights.pt", map_location="cpu", weights_only=True)
    model.topic_head.load_state_dict(topic_weights)
    model.harm_head.load_state_dict(harm_weights)

model.eval()
print("Model loaded successfully!")

Loading tokenizer and model...
Model loaded successfully!


In [9]:
def predict_clause(text: str, topic_threshold: float = 0.5):
    """Predicts topics and consumer harm for a given text clause."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding="max_length")
    
    with torch.no_grad():
        topic_logits, harm_logits = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs.get("token_type_ids")
        )
        
    # Process topic predictions
    topic_probs = torch.sigmoid(topic_logits).squeeze().tolist()
    predicted_topics = []
    for i, prob in enumerate(topic_probs):
        if prob >= topic_threshold:
            topic_name = id2name.get(i, id2label.get(i, f"Topic {i}"))
            predicted_topics.append((topic_name, prob))
            
    predicted_topics.sort(key=lambda x: x[1], reverse=True)
    
    # Process harm predictions
    harm_probs = torch.softmax(harm_logits, dim=-1).squeeze().tolist()
    harm_class_idx = torch.argmax(harm_logits, dim=-1).item()
    harm_label = HARM_CLASS_NAMES.get(harm_class_idx, "Unknown")
    harm_confidence = harm_probs[harm_class_idx]
    
    # Display results
    print("="*60)
    print("CLAUSE:")
    print(text)
    print("\n--- TOPIC PREDICTIONS ---")
    if not predicted_topics:
        print("No topics detected above threshold.")
    else:
        for name, prob in predicted_topics:
            print(f"{prob:.4f} | {name}")
            
    print("\n--- HARM PREDICTION ---")
    print(f"{harm_label} (Confidence: {harm_confidence:.4f})")
    print("All harm probabilities:")
    for idx, name in HARM_CLASS_NAMES.items():
        print(f"  {name}: {harm_probs[idx]:.4f}")
    print("="*60)

In [10]:
test_clause = "Solely for the purposes of operating or improving the Services and Software, you grant us a non-exclusive, worldwide, royalty-free sublicensable, license, to use, reproduce, publicly display, distribute, modify, create derivative works based on, publicly perform, and translate the Content. For example, Adobe may sublicense our right to the Content to our service providers or to other users to allow the Services and Software to operate as intended, such as enabling you to share photos with others."

predict_clause(test_clause)

CLAUSE:
Solely for the purposes of operating or improving the Services and Software, you grant us a non-exclusive, worldwide, royalty-free sublicensable, license, to use, reproduce, publicly display, distribute, modify, create derivative works based on, publicly perform, and translate the Content. For example, Adobe may sublicense our right to the Content to our service providers or to other users to allow the Services and Software to operate as intended, such as enabling you to share photos with others.

--- TOPIC PREDICTIONS ---
0.9605 | Copyright License

--- HARM PREDICTION ---
Harmful (Confidence: 0.9996)
All harm probabilities:
  Harmful: 0.9996
  Neutral: 0.0003
  Fair: 0.0001


In [11]:
test_clause_1 = "You may cancel your subscription and terminate your use of the Services and Software at any time. Cancellation or termination of your account does not relieve you of any obligation to pay any outstanding fees associated with your subscription, including, but not limited to early cancellation fees"

predict_clause(test_clause_1)

CLAUSE:
You may cancel your subscription and terminate your use of the Services and Software at any time. Cancellation or termination of your account does not relieve you of any obligation to pay any outstanding fees associated with your subscription, including, but not limited to early cancellation fees

--- TOPIC PREDICTIONS ---
0.8582 | Payments

--- HARM PREDICTION ---
Neutral (Confidence: 0.9280)
All harm probabilities:
  Harmful: 0.0693
  Neutral: 0.9280
  Fair: 0.0028


In [ ]:
test_clause_2 = "Adobe may access, view, or listen to your Content through both automated and manual methods, but only in limited ways, and only as permitted by law."

predict_clause(test_clause_2)

CLAUSE:
Adobe may access, view, or listen to your Content through both automated and manual methods

--- TOPIC PREDICTIONS ---
0.9838 | Content Rules
0.9387 | Content Removal

--- HARM PREDICTION ---
Neutral (Confidence: 0.9995)
All harm probabilities:
  Harmful: 0.0005
  Neutral: 0.9995
  Fair: 0.0000


In [ ]:
test_clause_3 = "NEITHER WE NOR ANY OF OUR AFFILIATES OR LICENSORS WILL BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR EXEMPLARY DAMAGES, INCLUDING DAMAGES FOR LOSS OF PROFITS, GOODWILL, USE, OR DATA OR OTHER LOSSES, EVEN IF WE HAVE BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES. OUR AGGREGATE LIABILITY UNDER THESE TERMS WILL NOT EXCEED ​​THE GREATER OF THE AMOUNT YOU PAID FOR THE SERVICE THAT GAVE RISE TO THE CLAIM DURING THE 12 MONTHS BEFORE THE LIABILITY AROSE OR ONE HUNDRED DOLLARS ($100). THE LIMITATIONS IN THIS SECTION APPLY ONLY TO THE MAXIMUM EXTENT PERMITTED BY APPLICABLE LAW."

predict_clause(test_clause_3)

CLAUSE:
NEITHER WE NOR ANY OF OUR AFFILIATES OR LICENSORS WILL BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR EXEMPLARY DAMAGES, INCLUDING DAMAGES FOR LOSS OF PROFITS, GOODWILL, USE, OR DATA OR OTHER LOSSES, EVEN IF WE HAVE BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES. OUR AGGREGATE LIABILITY UNDER THESE TERMS WILL NOT EXCEED ​​THE GREATER OF THE AMOUNT YOU PAID FOR THE SERVICE THAT GAVE RISE TO THE CLAIM DURING THE 12 MONTHS BEFORE THE LIABILITY AROSE OR ONE HUNDRED DOLLARS ($100). THE LIMITATIONS IN THIS SECTION APPLY ONLY TO THE MAXIMUM EXTENT PERMITTED BY APPLICABLE LAW.

--- TOPIC PREDICTIONS ---
0.9951 | Service Governance
0.9948 | Contract Formed Through Use
0.8909 | Discretionary Interpretation
0.8798 | Complaint Handling System

--- HARM PREDICTION ---
Harmful (Confidence: 0.9999)
All harm probabilities:
  Harmful: 0.9999
  Neutral: 0.0001
  Fair: 0.0000


In [16]:
test_clause_4 = "If you are a business or organization, to the extent permitted by law, you will indemnify and hold harmless us, our affiliates, and our personnel, from and against any costs, losses, liabilities, and expenses (including attorneys’ fees) from third party claims arising out of or relating to your use of the Services and Content or any violation of these Terms."

predict_clause(test_clause_4)

CLAUSE:
If you are a business or organization, to the extent permitted by law, you will indemnify and hold harmless us, our affiliates, and our personnel, from and against any costs, losses, liabilities, and expenses (including attorneys’ fees) from third party claims arising out of or relating to your use of the Services and Content or any violation of these Terms.

--- TOPIC PREDICTIONS ---
0.9995 | Service Governance
0.9995 | Contract Formed Through Use
0.9992 | Complaint Handling System
0.9985 | Discretionary Interpretation
0.9829 | Severability

--- HARM PREDICTION ---
Neutral (Confidence: 1.0000)
All harm probabilities:
  Harmful: 0.0000
  Neutral: 1.0000
  Fair: 0.0000


In [17]:
test_clause_5 = " You and OpenAI agree to resolve any claims arising out of or relating to these Terms or our Services, regardless of when the claim arose, even if it was before these Terms existed (a “Dispute”), through final and binding arbitration. You may opt out of arbitration within 30 days of account creation or of any updates to these arbitration terms within 30 days after the update has taken effect by filling out this form⁠. If you opt out of an update, the last set of agreed upon arbitration terms will apply. "

predict_clause(test_clause_5)

CLAUSE:
 You and OpenAI agree to resolve any claims arising out of or relating to these Terms or our Services, regardless of when the claim arose, even if it was before these Terms existed (a “Dispute”), through final and binding arbitration. You may opt out of arbitration within 30 days of account creation or of any updates to these arbitration terms within 30 days after the update has taken effect by filling out this form⁠. If you opt out of an update, the last set of agreed upon arbitration terms will apply. 

--- TOPIC PREDICTIONS ---
0.9387 | Mandatory Arbitration
0.9315 | Class Action Waiver

--- HARM PREDICTION ---
Harmful (Confidence: 0.9999)
All harm probabilities:
  Harmful: 0.9999
  Neutral: 0.0000
  Fair: 0.0000


In [18]:
test_clause_6 = "You and OpenAI agree that Disputes must be brought on an individual basis only, and may not be brought as a plaintiff or class member in any purported class, consolidated, or representative proceeding. Class arbitrations, class actions, and representative actions are prohibited. Only individual relief is available. The parties agree to sever and litigate in court any request for public injunctive relief after completing arbitration for the underlying claim and all other claims. This does not prevent either party from participating in a class-wide settlement. You and OpenAI knowingly and irrevocably waive any right to trial by jury in any action, proceeding, or counterclaim. "

predict_clause(test_clause_6)

CLAUSE:
You and OpenAI agree that Disputes must be brought on an individual basis only, and may not be brought as a plaintiff or class member in any purported class, consolidated, or representative proceeding. Class arbitrations, class actions, and representative actions are prohibited. Only individual relief is available. The parties agree to sever and litigate in court any request for public injunctive relief after completing arbitration for the underlying claim and all other claims. This does not prevent either party from participating in a class-wide settlement. You and OpenAI knowingly and irrevocably waive any right to trial by jury in any action, proceeding, or counterclaim. 

--- TOPIC PREDICTIONS ---
0.9216 | Class Action Waiver
0.8608 | Mandatory Arbitration

--- HARM PREDICTION ---
Harmful (Confidence: 1.0000)
All harm probabilities:
  Harmful: 1.0000
  Neutral: 0.0000
  Fair: 0.0000


In [19]:
test_clause_7 = "If you are a business or organization, to the extent permitted by law, you will indemnify and hold harmless us, our affiliates, and our personnel, from and against any costs, losses, liabilities, and expenses (including attorneys’ fees) from third party claims arising out of or relating to your use of the Services and Content or any violation of these Terms."

predict_clause(test_clause_7)

CLAUSE:
If you are a business or organization, to the extent permitted by law, you will indemnify and hold harmless us, our affiliates, and our personnel, from and against any costs, losses, liabilities, and expenses (including attorneys’ fees) from third party claims arising out of or relating to your use of the Services and Content or any violation of these Terms.

--- TOPIC PREDICTIONS ---
0.9995 | Service Governance
0.9995 | Contract Formed Through Use
0.9992 | Complaint Handling System
0.9985 | Discretionary Interpretation
0.9829 | Severability

--- HARM PREDICTION ---
Neutral (Confidence: 1.0000)
All harm probabilities:
  Harmful: 0.0000
  Neutral: 1.0000
  Fair: 0.0000


In [20]:
test_clause_8 = "You grant Apollo an irrevocable, perpetual, worldwide, transferable, sublicensable, and royalty-free license to analyze Customer Data using artificial intelligence to improve the Platform; and to test, develop, improve, or enhance Apollo’s products and services provided that Apollo will not refer to or associate Customer Data with any such analytics."

predict_clause(test_clause_8)

CLAUSE:
You grant Apollo an irrevocable, perpetual, worldwide, transferable, sublicensable, and royalty-free license to analyze Customer Data using artificial intelligence to improve the Platform; and to test, develop, improve, or enhance Apollo’s products and services provided that Apollo will not refer to or associate Customer Data with any such analytics.

--- TOPIC PREDICTIONS ---
No topics detected above threshold.

--- HARM PREDICTION ---
Harmful (Confidence: 1.0000)
All harm probabilities:
  Harmful: 1.0000
  Neutral: 0.0000
  Fair: 0.0000


In [21]:
test_clause_9 = "Subscriptions are non-cancelable during the Term and all payments by you are nonrefundable. There are no refunds for partially used Services or service units. In Apollo’s sole discretion, Apollo may elect to provide you with a refund, discount, or other consideration."

predict_clause(test_clause_9)

CLAUSE:
Subscriptions are non-cancelable during the Term and all payments by you are nonrefundable. There are no refunds for partially used Services or service units. In Apollo’s sole discretion, Apollo may elect to provide you with a refund, discount, or other consideration.

--- TOPIC PREDICTIONS ---
0.9915 | Payments

--- HARM PREDICTION ---
Neutral (Confidence: 1.0000)
All harm probabilities:
  Harmful: 0.0000
  Neutral: 1.0000
  Fair: 0.0000


In [22]:
test_clause_10 = "You agree to indemnify, defend and hold us, our affiliates, directors, officers, employees, contractors, and agents, and our suppliers, licensors, and service providers harmless from and against any actual or threatened loss, liability, claim, demand, damages, costs or expenses by a third party... arising out of or in connection with: (1) Your use of the Service; (2) Your breach of these Terms of Service; (3) Your violation of any applicable law or rights; or (4) the Customer Data."

predict_clause(test_clause_10)

CLAUSE:
You agree to indemnify, defend and hold us, our affiliates, directors, officers, employees, contractors, and agents, and our suppliers, licensors, and service providers harmless from and against any actual or threatened loss, liability, claim, demand, damages, costs or expenses by a third party... arising out of or in connection with: (1) Your use of the Service; (2) Your breach of these Terms of Service; (3) Your violation of any applicable law or rights; or (4) the Customer Data.

--- TOPIC PREDICTIONS ---
0.9994 | Service Governance
0.9994 | Contract Formed Through Use
0.9989 | Complaint Handling System
0.9977 | Discretionary Interpretation
0.9830 | Severability

--- HARM PREDICTION ---
Neutral (Confidence: 1.0000)
All harm probabilities:
  Harmful: 0.0000
  Neutral: 1.0000
  Fair: 0.0000
